# Lecture 15 notebook: who trades and what it costs

This notebook replays Lecture 15 on the tape of Lecture 12. Every figure and number of the lecture comes from a script in `scripts/`, next to this
notebook; each runs in a fresh process, so that memory is freed after every step.

| § | Content | Lecture |
|---|---|---|
| 0 | Settings | |
| 1 | Does one-sided flow predict the price? | §2 |
| 2 | Sweeps and absorption | §3 |
| 3 | Cascades: fuel and spark | §4 |
| 4 | What a trade costs | §5 |
| 5 | Resting orders: fills, misses, the cut-off and re-posting | §6 |
| 6 | Capacity: impact and funding | §7 |
| 7 | A test of your own | §9.2 |

**Before you start.** Lecture 12's notebook must have built the tape for October 2021 to September 2026. Results go to `data/interim/l15`;
figures are saved in `figures/` as `nb_*`, so the lecture's own figures are never overwritten. Three steps read or download large amounts of data and
are off by default (`RUN_HEAVY` in §0); with the flag off, the notebook uses their saved output in `data/interim`.

## 0. Settings

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path
import numpy as np, polars as pl, psutil
from IPython.display import Image, Markdown, display

HERE = Path.cwd()
SCRIPTS, FIG = HERE / "scripts", HERE / "figures"
sys.path[:0] = [str(SCRIPTS), str(HERE.parent / "13-bars-not-cut-by-time" / "scripts"), str(HERE.parent / "12-tick-data-the-tape"), str(HERE.parents[1])]
import tape
OUT = tape.ROOT / "interim" / "l15"

RUN_HEAVY = False      # True reruns: the extraction of all sweeps from the tape (about 40 minutes), the download of Binance's five-minute open interest
                       # (1,826 small files), and the best bid and ask of 80 days (8 GB of downloads, many hours); False uses their saved output

pl.Config.set_tbl_rows(40); pl.Config.set_fmt_str_lengths(80); pl.Config.set_tbl_width_chars(200)
print("data folder:", "the folder in the environment variable TAPE_ROOT" if os.environ.get("TAPE_ROOT") else "data/ at the top of the repository")
print(f"free memory: {psutil.virtual_memory().available / 1e9:.1f} GB of {psutil.virtual_memory().total / 1e9:.1f} GB")


def run(script, *args, show=False):
    """Run a script of scripts/ in a fresh process; figures to figures/ as nb_*, results to data/interim/l15."""
    t0 = time.time()
    env = {**os.environ, "TAPE_ROOT": str(tape.ROOT), "L15_PREFIX": "nb_", "PYTHONIOENCODING": "utf-8"}
    p = subprocess.run([sys.executable, "-I", str(SCRIPTS / script), *args], capture_output=True, text=True, encoding="utf-8", env=env)
    if p.returncode:
        print(p.stdout[-2000:], p.stderr[-4000:]); raise RuntimeError(f"{script} failed")
    print(f"{script}: {time.time() - t0:.0f} s")
    if show: print(p.stdout)


def result(name):
    return json.loads((OUT / name).read_text(encoding="utf-8"))


def pictures(*names):
    for nm in names:
        f = FIG / f"nb_{nm}.png"
        display(Image(filename=str(f))) if f.exists() else print("not drawn yet:", f.name)

## 1. Does one-sided flow predict the price?

Lecture §2. Windows of 1, 5 and 30 minutes sorted into fifths by net aggressive buying: the return during the window and over the next, by
volatility regime; the strongest flow split by its relation to the slower trend; and three scheduled releases. Figures 15_01 to 15_03.

In [ ]:
run("l15_flow.py"); run("l15_flow_matrix.py"); run("l15_s2_charts.py")
pictures("15_01_impact_and_after", "15_02_flow_matrix", "15_03_releases")
display(Markdown("**At all CPI and Fed releases: the first five minutes' flow and move against the next thirty minutes (correlations)**"))
pl.DataFrame([{"release": k, **v} for k, v in result("l15_s2_charts.json").items()])

## 2. Sweeps and absorption

Lecture §3. Every aggressive order that moved the price by 5 bp or more; the independent ones (first in each 10-minute block); the price path after
them, with and against the slower trend; the test against moments without a sweep, with days as clusters and the Benjamini-Hochberg correction.
Figure 15_04.

In [ ]:
if RUN_HEAVY:
    run("l15_sweeps.py")
run("l15_sweeps_matrix.py"); run("l15_sweeps_checks.py"); run("l15_s3_charts.py")
pictures("15_04_sweep_absorption")
display(Markdown("**Return in the trend's direction after the sweep's last fill, minus the regime-matched control, bp**"))
pl.DataFrame(result("l15_sweeps_checks.json")["pooled"]).select("min_bp", "block", "rel", "h", "n", "diff", "t", "q_bh")

## 3. Cascades: fuel and spark

Lecture §4. A cascade is an hour whose move, over its typical size, is in the top 1%. Ten warnings, each tested beyond the last hour's own volatility,
with cut-offs from 2021-2024 and a test on 2025-2026; then the checks on direction and extremes. Figures 15_05 and 15_06.

In [ ]:
if RUN_HEAVY:
    run("l15_metrics_download.py")
run("l15_cascades.py"); run("l15_cascades_checks.py"); run("l15_s4_charts.py")
pictures("15_05_risk_clusters", "15_06_warnings")
cas = result("l15_cascades.json")
display(Markdown(f"**Chance that the next hour is a cascade, % by fifth of the last hour's volatility:** {cas['control']}"))
pl.DataFrame(cas["warnings"]).select("warning", "horizon", "sample", "flagged", "lift", "added_pp", "t")

## 4. What a trade costs

Lecture §5. The best bid and ask of 80 days (every fourth day, May 2023 to March 2024), matched to every aggressive order: quoted and effective
half-spread, impact and realised spread at 1, 10 and 60 seconds, by order size and volatility regime. Figures 15_07 and 15_08.

In [ ]:
if RUN_HEAVY:
    run("l15_costs.py")
run("l15_costs_analysis.py"); run("l15_s5_charts.py")
pictures("15_07_cost_by_size", "15_08_adverse_selection")
co = result("l15_costs_analysis.json")
display(Markdown(f"**Days {co['days']}, one tick {co['time_share_one_tick']:.1%} of the time, mean quoted half-spread {co['time_mean_quoted_bp']} bp**"))
pl.DataFrame(co["by_size"]).select("bucket", "orders", "quoted", "eff", "imp_10s", "real_10s", "real_60s", "t_real_10s")

## 5. Resting orders

Lecture §6. At every seventh minute, a buy and a mirrored sell: a market order now, or a resting order that chases at a cut-off. Fill rates and the
saving by distance and wait (one-minute bars, five years); what the misses are; the cut-off and re-posting on the tick tape (every tenth day).
Figures 15_09 to 15_11.

In [ ]:
run("l15_resting.py"); run("l15_resting_tail.py"); run("l15_resting_repost.py"); run("l15_s6_charts.py")
pictures("15_09_resting_fill_and_saving", "15_10_resting_what_it_misses", "15_11_resting_cutoff")
display(Markdown("**The cut-off, tick tape: cost against a market order, bp**"))
display(pl.DataFrame(result("l15_resting_tail.json")))
display(Markdown("**Re-posted every few seconds**"))
pl.DataFrame(result("l15_resting_repost.json"))

## 6. Capacity: impact and funding

Lecture §7 and the appendix. Daily volume and volatility by year; the capacity map from the square-root law; funding against interest on the spare
cash; long spot against long perpetual (the spot candles are downloaded on first run); aggregate impact on the tape. Figures 15_12 to 15_15.

In [ ]:
run("l15_capacity.py"); run("l15_perp_vs_spot.py"); run("l15_s7_charts.py")
pictures("15_12_capacity", "15_13_funding", "15_14_perp_vs_spot", "15_15_aggregate_impact")
cap, pv = result("l15_capacity.json"), result("l15_perp_vs_spot.json")
display(pl.DataFrame(cap["by_year"])); display(pl.DataFrame(pv["by_year"]))
pv["growth"]

## 7. A test of your own

The re-posted entry of §6 on one day of your choice, with your own re-post interval and cut-off (exercise 4 of §9.2): at every seventh minute a buy
rests one tick below the last price, is re-posted every `STEP` seconds, and becomes a market order after `CUT` seconds.

In [ ]:
from datetime import date, datetime, timezone
from l15_resting_fn import first_through

DAY, STEP, CUT = date(2024, 3, 5), 5, 60          # change: any day of the tape, re-post interval and cut-off in seconds
a = int(datetime(DAY.year, DAY.month, DAY.day, tzinfo=timezone.utc).timestamp() * 1000)
t = (pl.scan_parquet(tape.source("BTCUSDT", DAY.strftime("%Y-%m"))).filter(pl.col("transact_time").is_between(a - 60_000, a + 86_400_000))
     .select("transact_time", "price").collect().sort("transact_time"))
ts, px = t["transact_time"].to_numpy(), t["price"].to_numpy()
dec = a + np.arange(60_000, 86_400_000 - CUT * 1000, 7 * 60_000); p0 = px[np.searchsorted(ts, dec, side="right") - 1]
cost = np.full(len(dec), np.nan); open_ = np.ones(len(dec), bool); s = 0
while open_.any() and s * STEP < CUT:
    idx = np.nonzero(open_)[0]; tk = dec[idx] + s * STEP * 1000
    lim = px[np.searchsorted(ts, tk, side="right") - 1] - 0.1
    hit = first_through(ts, px, tk, lim, 1, min(STEP, CUT - s * STEP) * 1000) >= 0
    cost[idx[hit]] = np.log(lim[hit] / p0[idx[hit]]) * 1e4 + 2.0; open_[idx[hit]] = False; s += 1
idx = np.nonzero(open_)[0]
cost[idx] = np.log(px[np.searchsorted(ts, dec[idx] + CUT * 1000, side="right") - 1] / p0[idx]) * 1e4 + 5.04
saving = cost - 5.04
pl.DataFrame({"decisions": [len(dec)], "sent as market order": [round(float(open_.mean()), 3)], "average against a market order, bp": [round(float(saving.mean()), 2)],
              "worst 5%, bp": [round(float(np.sort(saving)[-max(1, len(saving) // 20):].mean()), 1)]})

Negative is cheaper than a market order sent at once. Compare a calm day with a volatile one, and short re-post intervals with long ones: where does the saving stop growing, and where does the tail start?